In [1]:
!pip install wfdb torchmetrics

You should consider upgrading via the '/Users/sinchana/Documents/GitHub/cal-poly-csc/CSC-487/final-project/ECG-detection/ecg_env/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import pandas as pd
import numpy as np
import wfdb
import ast
import torch
from sklearn.preprocessing import MultiLabelBinarizer
import torchmetrics
import os

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [4]:
from config import PARENT_PATH, CSV_PATH, BEST_MODEL_SAVE_DIR

In [5]:
if device.type == 'cuda':
    from google.colab import drive
    drive.mount('/content/drive')

In [6]:
if device.type == 'cuda':
    !time unzip -q "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip" -d "/content/"
    !ls -lah /content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 | head

In [7]:
if device.type == 'cuda':
    best_model_save_dir = '/content/drive/MyDrive/'
else:
    best_model_save_dir = BEST_MODEL_SAVE_DIR

In [8]:
def load_raw_data(df, sampling_rate):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(os.path.join(parent_path, f)) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(os.path.join(parent_path, f)) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

if device.type == 'cuda':
    #parent_path = '/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
    parent_path = PARENT_PATH
else:
    #parent_path = os.curdir
    parent_path = PARENT_PATH
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(os.path.join(CSV_PATH, 'ptbxl_database.csv'), index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(os.path.join(CSV_PATH, 'scp_statements.csv'), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

In [9]:
# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

In [10]:
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(y_train)
y_test = mlb.transform(y_test)

In [11]:
X_train.shape, X_train.dtype, y_train.shape, y_train.dtype

((19601, 1000, 12), dtype('float64'), (19601, 5), dtype('int64'))

In [12]:
# just examining the data some
summed = np.sum(y_train, axis=1)
record_ids_with_no_labels_true = []
for id, class_sum in enumerate(summed):
    if class_sum == 0:
        record_ids_with_no_labels_true.append(id)
print(f'{len(record_ids_with_no_labels_true)} train examples have all labels as "0"')

371 train examples have all labels as "0"


In [13]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32).transpose(1,2)  # the 12 channels dimension is initially loaded in as last
y_train_tensor = torch.tensor(y_train,dtype=torch.float32)

X_test_tensor = torch.tensor(X_test,dtype=torch.float32).transpose(1,2)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32)

In [14]:
from torch.utils.data import TensorDataset,DataLoader
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [15]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv1 = nn.Conv1d(12, 32, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)

        # average the whole time series into 1 length to account for any time series length
        # while this discards temporal detail, the abnormalities are more concerned over the presence/strength of extracted features (representing the presence of abnormalities)
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.squeeze(-1)

        x = self.fc(x)

        return x

In [16]:
# TODO: try adding pos_weight with pos_weight_c = N_neg_c / N_pos_c (where c is each of the 5 labels)
# count class frequencies in training set
pos_counts = y_train_tensor.sum(dim=0)
neg_counts = len(y_train_tensor) - pos_counts
pos_weight = (neg_counts / pos_counts).to(device)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

loss_fn = nn.BCEWithLogitsLoss()  

In [17]:
def train_model(model, optimizer, loss_fn, train_loader, test_loader, best_model_save_path, num_epochs=10):
    best_loss = float('inf')  # start with infinity
    for epoch in range(num_epochs):
        running_train_loss = 0.0
        model.train()

        for x, y in train_loader:

            x = x.to(device)
            y = y.to(device)

            preds = model(x)

            loss = loss_fn(preds, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item() * x.size(0)
        epoch_train_loss = running_train_loss / len(train_loader.dataset)

        running_test_loss = 0.0
        model.eval()
        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                y = y.to(device)
                preds = model(x)
                loss = loss_fn(preds, y)
                running_test_loss += loss.item() * x.size(0)
            epoch_test_loss = running_test_loss / len(test_loader.dataset)

        print(f"Epoch {epoch+1}, Training Loss: {epoch_train_loss:.4f}, Testing Loss: {epoch_test_loss:.4f}")

        if epoch_test_loss < best_loss:
            best_loss = epoch_test_loss
            torch.save(model.state_dict(), best_model_save_path)

In [18]:
def run_inference(model, dataloader):
    model.eval()
    all_test_preds, all_test_labels = [], []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_test_preds.append(preds.cpu())
            all_test_labels.append(y.cpu())

    # Concatenate all batches
    all_test_preds = torch.cat(all_test_preds, dim=0)
    all_test_labels = torch.cat(all_test_labels, dim=0)

    return all_test_preds, all_test_labels

In [19]:
model = CNN(num_classes=y_train_tensor.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_model(model, optimizer, loss_fn, train_loader, test_loader, os.path.join(best_model_save_dir, 'cnn_best_model.pt'), num_epochs=30)

Epoch 1, Training Loss: 0.3618, Testing Loss: 0.3568
Epoch 2, Training Loss: 0.3121, Testing Loss: 0.3243
Epoch 3, Training Loss: 0.2951, Testing Loss: 0.3120
Epoch 4, Training Loss: 0.2859, Testing Loss: 0.3091
Epoch 5, Training Loss: 0.2786, Testing Loss: 0.2999
Epoch 6, Training Loss: 0.2735, Testing Loss: 0.3082
Epoch 7, Training Loss: 0.2692, Testing Loss: 0.2970
Epoch 8, Training Loss: 0.2654, Testing Loss: 0.3226
Epoch 9, Training Loss: 0.2618, Testing Loss: 0.2902
Epoch 10, Training Loss: 0.2571, Testing Loss: 0.2923
Epoch 11, Training Loss: 0.2557, Testing Loss: 0.2898
Epoch 12, Training Loss: 0.2532, Testing Loss: 0.2833
Epoch 13, Training Loss: 0.2510, Testing Loss: 0.2935
Epoch 14, Training Loss: 0.2476, Testing Loss: 0.2884
Epoch 15, Training Loss: 0.2448, Testing Loss: 0.3105
Epoch 16, Training Loss: 0.2423, Testing Loss: 0.2832
Epoch 17, Training Loss: 0.2394, Testing Loss: 0.2945
Epoch 18, Training Loss: 0.2382, Testing Loss: 0.3031
Epoch 19, Training Loss: 0.2360, Test

In [21]:
all_test_preds, all_test_labels = run_inference(model, test_loader)

In [22]:
# Per-class accuracy
def calc_per_class_accs(all_test_preds, all_test_labels):
    per_class_acc = (all_test_preds == all_test_labels).float().mean(dim=0)  # [num_classes]
    # Print results
    for idx, acc in enumerate(per_class_acc):
        print(f"Class {mlb.classes_[idx]} accuracy: {acc.item():.3f}")
    print('\n')

In [23]:
from sklearn.metrics import f1_score
def calc_f1_scores(all_test_labels, all_test_preds, average=None):
    f1_scores = f1_score(all_test_labels, all_test_preds, average=None)
    for idx, score in enumerate(f1_scores):
        print(f"Class {mlb.classes_[idx]} F1-score: {score:.3f}")
    print('\n')

In [24]:
from sklearn.metrics import roc_auc_score

def calc_roc_auc_scores(all_test_labels, all_test_preds):
    num_classes = all_test_labels.shape[1]
    aucs = []

    for i in range(num_classes):
        try:
            auc = roc_auc_score(all_test_labels[:, i], all_test_preds[:, i])
        except ValueError:
            # happens if class i has no positive samples in test set
            auc = float('nan')
        aucs.append(auc)

    for idx, auc in enumerate(aucs):
        print(f"Class {mlb.classes_[idx]} AUROC: {auc:.3f}")
    print('\n')

In [25]:
print('-----CNN Test Evals-----')
calc_per_class_accs(all_test_labels, all_test_preds)
calc_f1_scores(all_test_labels, all_test_preds)
calc_roc_auc_scores(all_test_labels, all_test_preds)

-----CNN Test Evals-----
Class CD accuracy: 0.889
Class HYP accuracy: 0.916
Class MI accuracy: 0.869
Class NORM accuracy: 0.864
Class STTC accuracy: 0.875


Class CD F1-score: 0.723
Class HYP F1-score: 0.543
Class MI F1-score: 0.683
Class NORM F1-score: 0.851
Class STTC F1-score: 0.753


Class CD AUROC: 0.802
Class HYP AUROC: 0.701
Class MI AUROC: 0.767
Class NORM AUROC: 0.867
Class STTC AUROC: 0.850




### --- GRU Model ---

In [26]:
# GRU wants batched input as (B, L, H_in), unlike for our CNN model
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.float32)

X_test_tensor = torch.tensor(X_test,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32)

In [27]:
from torch.utils.data import TensorDataset,DataLoader
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [28]:
import torch.nn as nn
import torch.nn.functional as F

# following kaggle tutorial here https://www.kaggle.com/code/fanbyprinciple/learning-pytorch-3-coding-an-rnn-gru-lstm
class ECGGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, sequence_length, bidirectional):
        super(ECGGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bias=True,
            batch_first=True,
            bidirectional=bidirectional,
        )
        #self.fc1 = nn.Linear(hidden_size*sequence_length*(1+bidirectional), num_classes)  # if bidirectional, the output size is doubled, so (1+bidrectional) accounts for that
        self.fc1 = nn.Linear(hidden_size * (2 if bidirectional else 1), num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers * (2 if self.bidirectional else 1), x.size(0), self.hidden_size).to(device)
        out, _ = self.gru(x, h0)
        out = out[:, -1, :]  # only last timestep
        out = self.fc1(out)
        return out
    # def forward(self, x):
    #     h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
    #     out, _ = self.gru(x, h0)
    #     out = out.reshape(out.shape[0], -1)
    #     out = self.fc1(out)
    #     return out

takes about an hour to train

In [29]:
model = ECGGRU(
    input_size=12,
    hidden_size=64,
    num_layers=2,
    num_classes=5,
    sequence_length=1000,
    bidirectional=False  # keep False on CPU, True only on GPU
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_model(model, optimizer, loss_fn, train_loader, test_loader, os.path.join(best_model_save_dir, 'gru_best_model.pt'), num_epochs=30)

Epoch 1, Training Loss: 0.5130, Testing Loss: 0.4622
Epoch 2, Training Loss: 0.3847, Testing Loss: 0.3695
Epoch 3, Training Loss: 0.3336, Testing Loss: 0.3392
Epoch 4, Training Loss: 0.3107, Testing Loss: 0.3264
Epoch 5, Training Loss: 0.2972, Testing Loss: 0.3191
Epoch 6, Training Loss: 0.2877, Testing Loss: 0.3037
Epoch 7, Training Loss: 0.2805, Testing Loss: 0.2999
Epoch 8, Training Loss: 0.2731, Testing Loss: 0.3057
Epoch 9, Training Loss: 0.2675, Testing Loss: 0.3111
Epoch 10, Training Loss: 0.2611, Testing Loss: 0.2909
Epoch 11, Training Loss: 0.2569, Testing Loss: 0.2953
Epoch 12, Training Loss: 0.2525, Testing Loss: 0.2877
Epoch 13, Training Loss: 0.2491, Testing Loss: 0.2944
Epoch 14, Training Loss: 0.2452, Testing Loss: 0.2991
Epoch 15, Training Loss: 0.2413, Testing Loss: 0.2870
Epoch 16, Training Loss: 0.2376, Testing Loss: 0.2992
Epoch 17, Training Loss: 0.2351, Testing Loss: 0.2970
Epoch 18, Training Loss: 0.2320, Testing Loss: 0.2886
Epoch 19, Training Loss: 0.2284, Test

In [30]:
def run_inference_gru(model, dataloader):
    model.eval()
    all_test_preds, all_test_labels = [], []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_test_preds.append(preds.cpu())
            all_test_labels.append(y.cpu())

    # Concatenate all batches
    all_test_preds = torch.cat(all_test_preds, dim=0)
    all_test_labels = torch.cat(all_test_labels, dim=0)

    return all_test_preds, all_test_labels

In [32]:
# load the best model from GRU training
best_gru_model = ECGGRU(
    input_size=12,
    hidden_size=64,      # match what was trained
    num_layers=2,        # match what was trained
    num_classes=5,
    sequence_length=1000,
    bidirectional=False
).to(device)

checkpoint_path = os.path.join(BEST_MODEL_SAVE_DIR, 'gru_best_model.pt')
best_gru_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

<All keys matched successfully>

In [33]:
all_test_preds, all_test_labels = run_inference_gru(best_gru_model, test_loader)

In [ ]:
def calc_f1_scores(all_test_labels, all_test_preds, average=None):
    # Handle torch tensors
    if hasattr(all_test_labels, 'detach'):
        all_test_labels = all_test_labels.detach().cpu().numpy()
    if hasattr(all_test_preds, 'detach'):
        all_test_preds = all_test_preds.detach().cpu().numpy()
    
    f1_scores = f1_score(all_test_labels, all_test_preds, average=None)
    for idx, score in enumerate(f1_scores):
        print(f"Class {mlb.classes_[idx]} F1-score: {score:.3f}")

In [34]:
print('-----GRU Test Evals-----')
calc_per_class_accs(all_test_labels, all_test_preds)
calc_f1_scores(all_test_labels, all_test_preds)
calc_roc_auc_scores(all_test_labels, all_test_preds)

-----GRU Test Evals-----
Class CD accuracy: 0.879
Class HYP accuracy: 0.913
Class MI accuracy: 0.861
Class NORM accuracy: 0.868
Class STTC accuracy: 0.882


Class CD F1-score: 0.694
Class HYP F1-score: 0.470
Class MI F1-score: 0.683
Class NORM F1-score: 0.853
Class STTC F1-score: 0.746


Class CD AUROC: 0.784
Class HYP AUROC: 0.658
Class MI AUROC: 0.773
Class NORM AUROC: 0.869
Class STTC AUROC: 0.830


